In [1]:
%load_ext autoreload
%autoreload 2
import math
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import torch

from tqdm.notebook import tqdm

from scFM_density_estimation.models import *
from scFM_density_estimation.datamodules import *
from scFM_density_estimation.utils import *

In [24]:
def prepare_batch(X, C, num_classes, batch_size, device):
    indices = np.random.randint(X.shape[0], size=batch_size)
    x1 = torch.from_numpy(X[indices]).float().to(device)
    cond = torch.nn.functional.one_hot(torch.from_numpy(C[indices]).long(),
                                       num_classes=num_classes).float().to(device)
    return x1, cond

def weighted_wasserstein(X, C, num_classes, model):
    ws_dist = 0
    true_labels = np.argmax(C.cpu().numpy(), axis=1)
    labels = np.unique(true_labels)
    
    for label in labels:
        mask = (true_labels == label)
        generated_samples = model.run_simulation(X[mask], C[mask], n_steps=100)
        ws_dist += wasserstein(X[mask], generated_samples) * np.sum(mask)

    return ws_dist / C.shape[0]

def train_model(X, C, batch_size, hidden_layers):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cond_dim = 8
    dim = 50
    model = ConditionalFlowMatching(input_dim=dim, hidden_dims=hidden_layers,
                                       cond_dim=cond_dim, use_encoder=False,
                                       use_ot_sampler=False).to(device)
    optimizer = model.configure_optimizers()

    best_ws = 1e10
    step = 0
    
    for k in tqdm(range(200000)):
        optimizer.zero_grad()
    
        x1, cond = prepare_batch(X, C, cond_dim, batch_size, device)
        loss = model.shared_step(x1, cond)
        
        loss.backward()
        optimizer.step()
        
        if (k + 1) % 10000 == 0:
            x1, cond = prepare_batch(X, C, cond_dim, 10000, device)
            ws_dist = weighted_wasserstein(x1, cond, cond_dim, model)

            if ws_dist < best_ws:
                best_ws = ws_dist
                step = k
                torch.save(model.state_dict(), f"./weights/combosciplex_{hidden_layers}_{batch_size}.ckpt")

    print(f"The lowest ws for {hidden_layers} and {batch_size} is {best_ws}, it occured at step {step+1}.")

In [3]:
adata = sc.read_h5ad("./data/combosciplex.h5ad")
label = "Drug1"
X = adata.obsm["X_pca"]
C = adata.obs[label].cat.codes.values.copy()

In [26]:
for hidden_layers in [[64, 64, 64]]:
    for batch_size in [2048]:
        train_model(X, C, batch_size, hidden_layers)

  0%|          | 0/200000 [00:00<?, ?it/s]

The lowest ws for [64, 64, 64] and 2048 is 8.766383940253828, it occured at step 180000.


Model is 4x1024, no ot, 2048 batch size with ws 8.36609. \
Model is 3x512, no ot, 2048 batch size with ws 8.55065.

In [19]:
torch.nn.functional.one_hot(torch.from_numpy(np.array([(1, 5), (2, 5)])), num_classes=20)

tensor([[[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]],

        [[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]])